## Installing necessary libraries and Importing Data set 

In [ ]:
!pip install kaggle

In [ ]:
!kaggle datasets list

In [ ]:
!kaggle datasets download -d lakshmi25npathi/online-retail-dataset

In [ ]:
import zipfile

with zipfile.ZipFile("online-retail-dataset.zip",'r') as zip_ref:
    zip_ref.extractall("data")
    

## Load into pandas

In [ ]:
import pandas as pd

df = pd.read_excel("data/online_retail_II.xlsx")
df.head()

In [ ]:
xls = pd.ExcelFile("data/online_retail_II.xlsx")
print(xls.sheet_names)

## Use Latest Year Sheet

In [ ]:

df = pd.read_excel("data/online_retail_II.xlsx", sheet_name='Year 2010-2011')
df.head()

## Data Understanding

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

## Standardize Column Names

In [ ]:
df.columns = df.columns.str.strip().str.replace(' ','_')
df.columns

## Remove missing customer ID

In [ ]:
df = df.dropna(subset=['Customer_ID'])

## Remove Invalid Transactions

In [ ]:
# Remove negative or zero quantity
df = df[df['Quantity']>0]

# Remove zero or negative price
df = df[df['Price']>0]

In [ ]:
df['Customer_ID'] = df['Customer_ID'].astype(int)

## Create Revenue Column

In [ ]:
df['TotalPrice'] = df['Quantity']*df['Price']

## Handle Description Nulls

In [ ]:
df = df.dropna(subset=['Description'])

## Final Validation

In [ ]:
df.info()
df.describe()

In [ ]:
df.shape

## Import pyodbc library

In [ ]:
import pyodbc
print(pyodbc.drivers())

## Create Connection

In [ ]:
from sqlalchemy import create_engine
server = r"localhost\SQLEXPRESS"


engine = create_engine(
    f"mssql+pyodbc://@{server}/OnlineRetailDB?trusted_connection=yes&driver=ODBC+Driver+17+for+SQL+Server"
)

## Optimize data types for SQL Server

In [ ]:

from sqlalchemy import  types

dtype_mapping = {
    'Invoice': types.String(20),
    'Stockcode': types.String(20),
    'Description': types.String(255),
    'Quantity': types.Integer(),
    'InvoiceDate': types.DateTime(),
    'Price': types.Numeric(10,2),
    'Customer_ID': types.Integer(),
    'Country': types.String(50),
    'TotalPrice': types.Numeric(12,2)
}
                               


## Push Data to SQL Server

In [ ]:


df.to_sql(
    name="OnlineRetail",
    con=engine,
    if_exists='replace',
    index=False,
    dtype=dtype_mapping,
    chunksize=10000   # Important for performance
)
    